# 03 — Ekstraksi Fitur dan Input Model

Notebook ini mensyaratkan output privat prapemrosesan. Manifest, provenance dataset, laporan validasi dan hash-nya, hash sampel, jumlah baris, tipe/label, ID, dan group_key diperiksa lebih dulu. Artefak Parquet lama di direktori output dibersihkan sebelum setiap eksekusi. Jika upstream blocked atau pemeriksaan gagal, notebook mencatat blocked tanpa memuat tokenizer atau membaca teks sampel.

Jika input valid, teks asli ditokenisasi dengan indolem/indobertweet-base-uncased pada revisi tetap, special tokens aktif, truncation aktif, max_length 512, dan padding=False. Belum ada fold. input_ids dan attention_mask menjadi satu-satunya kolom input model; target dan metadata split tersimpan terpisah.

Output /kaggle/working/feature_extraction adalah output privat. Array token adalah turunan data dan tidak boleh dipublikasikan sebelum hak penggunaan dipastikan.

In [ ]:
import hashlib, json, unicodedata
from pathlib import Path
OUT = Path("/kaggle/working/feature_extraction"); OUT.mkdir(parents=True, exist_ok=True)
INPUT = Path("/kaggle/input")
REPORT, MANIFEST = OUT / "feature_report.json", OUT / "feature_manifest.json"
REPO = "Exqrch/IndoDiscourse"
REVISION = "8af8c0f00b05633c05cc41fbd7650178ababc892"
FILENAME = "indotoxic2024_annotated_data_v2_final.jsonl"
EXPECTED_SHA = "94bb43546f5040a606a8d842694b7d490df043051377a6580e1e95ad2c7334e7"
EXPECTED_ROWS = 28448
EXPECTED_COLUMNS = {"row_id", "text_id", "text", "toxicity_label", "polarized_label", "group_key"}
EXPECTED_SOURCE = {"repository": REPO, "configuration": "main", "revision": REVISION,
                   "filename": FILENAME, "sha256": EXPECTED_SHA,
                   "expected_rows": EXPECTED_ROWS, "actual_rows": EXPECTED_ROWS}

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

def clear_artifacts():
    for path in OUT.glob("*.parquet"):
        path.unlink(missing_ok=True)
    REPORT.unlink(missing_ok=True)
    MANIFEST.unlink(missing_ok=True)

def block(reason, details=None):
    clear_artifacts()
    write_json(REPORT, {"status": "blocked", "block_reason": reason,
        "details": details or {}, "tokenizer_loaded": False, "tokenization_performed": False})
    write_json(MANIFEST, {"status": "blocked", "report": REPORT.name})
    print("STATUS=blocked:", reason, "| tokenizer tidak dimuat.")

def read_json(path):
    try:
        return json.loads(path.read_text(encoding="utf-8")), None
    except (OSError, UnicodeError, json.JSONDecodeError) as exc:
        return None, type(exc).__name__

def resolve_input_file(root, parent, name):
    if not isinstance(name, str) or not name:
        return None
    try:
        path = (parent / name).resolve(strict=True)
        return path if root in path.parents and path.is_file() else None
    except (OSError, RuntimeError, ValueError):
        return None

def has_hash(path, expected):
    if not isinstance(expected, str):
        return False
    try:
        return sha256(path) == expected
    except OSError:
        return False

def validate_sample(sample, artifact, validation):
    import pandas as pd
    if len(sample) != artifact.get("row_count") or len(sample) != validation.get("sample_rows"):
        return "upstream_sample_row_count_mismatch"
    if set(sample.columns) != EXPECTED_COLUMNS:
        return "upstream_sample_schema_invalid"
    columns = artifact.get("columns")
    if not isinstance(columns, list) or not all(isinstance(col, str) for col in columns):
        return "upstream_manifest_sample_columns_invalid"
    if set(columns) != EXPECTED_COLUMNS:
        return "upstream_manifest_sample_columns_mismatch"
    if not sample["text"].map(lambda value: isinstance(value, str)).all():
        return "upstream_text_type_invalid"
    if not sample["text_id"].map(lambda value: isinstance(value, str) and bool(value.strip())).all():
        return "upstream_text_id_invalid"
    row_ids = sample["row_id"]
    if (not pd.api.types.is_integer_dtype(row_ids) or row_ids.isna().any()
            or row_ids.duplicated().any() or (row_ids < 0).any()
            or (row_ids >= EXPECTED_ROWS).any()):
        return "upstream_row_id_invalid"
    for col in ("toxicity_label", "polarized_label"):
        labels = sample[col]
        if (not pd.api.types.is_integer_dtype(labels) or labels.isna().any()
                or not labels.isin([0, 1]).all()):
            return "upstream_label_invalid"
    def make_group_key(text):
        normalized = unicodedata.normalize("NFKC", text).casefold()
        return " ".join(normalized.split())
    if not sample.apply(lambda row: isinstance(row["group_key"], str)
                        and row["group_key"] == make_group_key(row["text"]), axis=1).all():
        return "upstream_group_key_invalid"
    return None

clear_artifacts()
ready, sample, upstream_hash = False, None, None
reason, details = None, {}
candidates = sorted(INPUT.glob("**/preprocessing_manifest.json")) if INPUT.exists() else []
if len(candidates) != 1:
    reason, details = "upstream_manifest_missing_or_ambiguous", {"candidate_count": len(candidates)}
else:
    root = INPUT.resolve()
    manifest_path = resolve_input_file(root, candidates[0].parent, candidates[0].name)
    if manifest_path is None:
        reason = "upstream_manifest_path_invalid"
    else:
        upstream, error = read_json(manifest_path)
        if error:
            reason, details = "upstream_manifest_unreadable", {"error_type": error}
        elif not isinstance(upstream, dict):
            reason = "upstream_manifest_schema_invalid"
        elif upstream.get("status") != "ready":
            reason, details = "upstream_manifest_not_ready", {"upstream_status": upstream.get("status")}
        elif upstream.get("dataset") != EXPECTED_SOURCE:
            reason = "upstream_dataset_provenance_mismatch"
        else:
            validation_path = resolve_input_file(root, manifest_path.parent, upstream.get("validation_report"))
            if validation_path is None:
                reason = "upstream_validation_report_missing_or_invalid"
            elif not has_hash(validation_path, upstream.get("validation_report_sha256")):
                reason = "upstream_validation_report_hash_mismatch"
            else:
                validation, error = read_json(validation_path)
                if error or not isinstance(validation, dict):
                    reason, details = "upstream_validation_report_unreadable", {"error_type": error}
                elif (validation.get("status") != "ready" or validation.get("source") != EXPECTED_SOURCE
                      or validation.get("issue_count") != 0):
                    reason = "upstream_validation_report_not_ready"
                else:
                    artifact = upstream.get("sample_artifact")
                    if not isinstance(artifact, dict):
                        reason = "upstream_sample_artifact_manifest_invalid"
                    else:
                        sample_path = resolve_input_file(root, manifest_path.parent, artifact.get("filename"))
                        if sample_path is None:
                            reason = "upstream_sample_missing_or_invalid"
                        elif not has_hash(sample_path, artifact.get("sha256")):
                            reason = "upstream_sample_hash_mismatch"
                        else:
                            try:
                                import pandas as pd
                                sample = pd.read_parquet(sample_path)
                            except Exception as exc:
                                sample = None
                                reason, details = "upstream_sample_unreadable", {"error_type": type(exc).__name__}
                            if sample is not None:
                                reason = validate_sample(sample, artifact, validation)
                                if reason is None:
                                    ready, upstream_hash = True, artifact["sha256"]
if not ready:
    block(reason or "upstream_validation_failed", details=details)
else:
    print("Provenance, validation report, hashes, jumlah baris, dan skema upstream valid.")


## Pemeriksaan upstream

Metadata menghubungkan notebook 02 sebagai kernel source privat. Harus ditemukan tepat satu manifest ready dengan provenance dan laporan validasi yang cocok, file sampel dengan hash/jumlah baris/skema yang sesuai, label biner, row_id unik, serta group_key yang konsisten. Dengan manifest blocked saat ini, tokenisasi dilewati.

**Implikasi untuk metodologi.** Validasi mencegah feature extraction melewati isu data yang belum diputuskan. Tokenizer baru dimuat setelah upstream lolos.

In [ ]:
if ready:
    import importlib.metadata, platform
    import numpy as np
    from transformers import AutoTokenizer
    TOKENIZER_REPO = "indolem/indobertweet-base-uncased"
    TOKENIZER_REVISION = "32e28c05b47e33b6675d2670a1745c50a65e987a"
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_REPO, revision=TOKENIZER_REVISION, use_fast=True)
    ids, masks, lengths = [], [], []
    for start in range(0, len(sample), 128):
        encoded = tokenizer(sample["text"].iloc[start:start + 128].tolist(),
            add_special_tokens=True, truncation=True, max_length=512, padding=False,
            return_attention_mask=True)
        ids.extend(encoded["input_ids"]); masks.extend(encoded["attention_mask"])
        lengths.extend(len(row) for row in encoded["input_ids"])
    model_inputs = pd.DataFrame({"input_ids": ids, "attention_mask": masks})
    targets = sample[["row_id", "toxicity_label", "polarized_label"]].reset_index(drop=True)
    split_metadata = sample[["row_id", "text_id", "group_key"]].reset_index(drop=True)
    frames = {"model_inputs": model_inputs, "targets": targets, "split_metadata": split_metadata}
    artifacts = {}
    for key, frame in frames.items():
        path, tmp = OUT / f"{key}.parquet", OUT / f"{key}.tmp.parquet"
        frame.to_parquet(tmp, index=False); tmp.replace(path)
        artifacts[key] = {"filename": path.name, "sha256": sha256(path),
                          "row_count": len(frame), "columns": list(frame.columns)}
    def version(name):
        try:
            return importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            return None
    report = {
        "status": "ready", "row_count": len(sample),
        "tokenizer": {"repository": TOKENIZER_REPO, "revision": TOKENIZER_REVISION},
        "tokenization": {"add_special_tokens": True, "truncation": True, "max_length": 512,
                         "padding": False, "attention_mask": True,
                         "length_median": float(np.median(lengths)) if lengths else None,
                         "length_p95": float(np.quantile(lengths, .95)) if lengths else None},
        "model_input_columns": ["input_ids", "attention_mask"],
        "targets_columns": list(targets.columns), "split_metadata_columns": list(split_metadata.columns),
        "folds_created": False,
        "versions": {"python": platform.python_version(), "pandas": pd.__version__,
                     "numpy": np.__version__, "transformers": version("transformers"),
                     "tokenizers": version("tokenizers")},
        "artifacts": artifacts}
    write_json(REPORT, report)
    write_json(MANIFEST, {"status": "ready", "upstream_sample_sha256": upstream_hash,
        "row_count": len(sample), "artifacts": artifacts,
        "report": REPORT.name, "report_sha256": sha256(REPORT)})
    print("STATUS=ready: input, target, dan metadata split tersimpan terpisah.")

## Skema output

- model_inputs.parquet hanya memuat input_ids dan attention_mask.
- targets.parquet memuat row_id dan dua label.
- split_metadata.parquet memuat row_id, text_id, dan group_key untuk split mendatang; fold tidak dibuat.
- Report dan manifest mencatat skema, hash, jumlah baris, parameter tokenisasi, dan versi runtime.

Target, ID, group_key, fitur diagnostik, demografi, dan metadata sumber tidak digabung ke input model. Semua file tetap menjadi output privat Kaggle.